# Experimentos do artigo — H1, H2 e H3 (Colab gratuito, GPU T4)

**Antes de rodar:** Ambiente de execução → Alterar o tipo → **GPU T4**. Depois: **Ambiente de execução → Executar tudo**.

| Parte | O que faz | Tempo aproximado |
|---|---|---|
| 1–2 | Repositório, Drive e dados (RePro) | 2 min |
| 3 | **H1:** BERTimbau com rótulos humanos (H, 3 sementes) e controle só com a validação humana (V) | 15 min |
| 4 | **H3:** aplica o modelo H à Olist (metade "relato") e compara P(entrega negativa) com o atraso real | 5 min |
| 5 | **H2:** o Qwen2.5-7B-Instruct (4 bits, gratuito) rotula a validação e o treino do RePro | 2–3 h |
| 6 | **H2:** BERTimbau com os rótulos do LLM (L, 3 sementes), no mesmo teste humano | 10 min |
| 7–8 | Resumo de todas as condições e download de `resultados.zip` | 1 min |

Tudo fica salvo no Google Drive (`MyDrive/pln-absa-olist/`). **Se a sessão cair ou a cota acabar, abra de novo e clique em Executar tudo: o que já terminou é pulado e a rotulagem do LLM continua de onde parou.**

In [ ]:
# 1) Repositório, pacote e Drive
import os, sys
from pathlib import Path
if "google.colab" in sys.modules:
    !git clone -q https://github.com/vnevesl/pln-absa-olist.git 2>/dev/null || git -C pln-absa-olist pull -q
    %cd /content/pln-absa-olist
    !pip -q install -e ".[llm]"
    from google.colab import drive
    try:
        drive.mount("/content/drive")
        BASE = Path("/content/drive/MyDrive/pln-absa-olist")
    except Exception as erro:  # permissão negada ou parcial: segue sem o Drive
        print(f"⚠️ Drive não conectou ({erro}). NÃO feche a aba até baixar o zip no fim.")
        BASE = Path("/content/pln-absa-olist-saida")
else:
    BASE = Path("results/colab_local")
# A instalação editável só vale depois de reiniciar o Python; isto torna o pacote visível já nesta sessão
_src = str(Path("src").resolve())
if _src not in sys.path:
    sys.path.insert(0, _src)
SAIDA = BASE / "resultados"; SAIDA.mkdir(parents=True, exist_ok=True)
MODELOS = BASE / "modelos"; MODELOS.mkdir(parents=True, exist_ok=True)  # fora do zip de resultados
CACHE = BASE / "llm_cache"; CACHE.mkdir(parents=True, exist_ok=True)
CKPT = Path("/content/ckpt" if "google.colab" in sys.modules else BASE / "ckpt")  # disco local, rápido
CKPT.mkdir(parents=True, exist_ok=True)
Path("data").mkdir(exist_ok=True)
if not Path("data/llm_cache").exists():
    os.symlink(CACHE, "data/llm_cache")  # respostas do LLM vão direto para o Drive
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NENHUMA — troque o ambiente para GPU T4!")

In [ ]:
# 2) Dados: RePro; a validação é dividida em exemplos do prompt (demo) e avaliação (val_dev)
import gc, json, time
import pandas as pd
from sklearn.model_selection import train_test_split
from absa.config import ASPECTS, LABEL_COLUMNS, POLARITIES, SEED
from absa.metrics import bootstrap_ci, detection_f1, macro_f1, per_aspect_report, tuple_f1
from absa.models.encoder import ENCODERS, EncoderAspectClassifier, TrainConfig
from absa.preprocess import clean_light
from absa.repro import load_repro

treino, val, teste, _ = load_repro()
demo, val_dev = train_test_split(val, train_size=200, stratify=val.nota, random_state=SEED)
demo, val_dev = demo.reset_index(drop=True), val_dev.reset_index(drop=True)
entrada = lambda df: df.texto_modelo.map(clean_light).tolist()
print(f"treino {len(treino)} · val_dev {len(val_dev)} · demo {len(demo)} · teste {len(teste)}")

def treinar(cond, seed, X, y, Xd, yd, salvar_modelo=False):
    """Treina o BERTimbau, escolhe a época em (Xd, yd), prediz o teste humano e salva no Drive."""
    pred_arq = SAIDA / f"pred_teste_bertimbau_{cond}_s{seed}.csv"
    if pred_arq.exists():
        print(f"✓ {cond} s{seed} já feito — pulando"); return
    t0 = time.time()
    clf = EncoderAspectClassifier(TrainConfig(model_name=ENCODERS["bertimbau"], epochs=4, lr=3e-5,
                                              batch_size=32, max_len=128, seed=seed))
    ckpt = CKPT / f"{cond}_s{seed}.pt"
    clf.fit(X, y, Xd, yd, save_path=ckpt)
    clf.load(ckpt)
    proba = clf.predict_proba(entrada(teste))
    pred = pd.DataFrame({c: [POLARITIES[i] for i in proba[:, j].argmax(1)] for j, c in enumerate(LABEL_COLUMNS)})
    pred.insert(0, "review_id", teste.review_id.values)
    pred.to_csv(pred_arq, index=False)
    json.dump({"cond": cond, "seed": seed, "n_treino": len(X), "minutos": (time.time() - t0) / 60,
               "historico": clf.history}, open(SAIDA / f"log_bertimbau_{cond}_s{seed}.json", "w"), indent=1)
    if salvar_modelo:
        torch.save(clf.model.state_dict(), MODELOS / f"bertimbau_{cond}_s{seed}.pt")
    print(f"→ {cond} s{seed}: F1 de pares {tuple_f1(teste, pred):.3f} ({(time.time() - t0) / 60:.1f} min)")
    ckpt.unlink(missing_ok=True)
    del clf; torch.cuda.empty_cache()

In [ ]:
# 3) H1 — BERTimbau com rótulos humanos (H, 3 sementes) e controle só com a validação humana (V)
for seed in [13, 42, 87]:
    treinar("H", seed, entrada(treino), treino, entrada(val_dev), val_dev, salvar_modelo=(seed == 42))
treinar("V", 42, entrada(val_dev), val_dev, entrada(demo), demo)  # V: época escolhida no demo

In [ ]:
# 4) H3 — Olist (metade "relato"): P(entrega negativa) × atraso real
from sklearn.metrics import roc_auc_score
from absa.data import load_reviews
from absa.sampling import gold_exclusion, olist_split

arq_olist = SAIDA / "olist_relato_probs_bertimbau_H_s42.csv"
if not arq_olist.exists():
    olist = load_reviews()
    olist = olist[olist.entregue & (olist_split(olist) == "relato") & ~gold_exclusion(olist)].reset_index(drop=True)
    clf = EncoderAspectClassifier(TrainConfig(model_name=ENCODERS["bertimbau"], max_len=128, batch_size=64))
    clf.model.load_state_dict(torch.load(MODELOS / "bertimbau_H_s42.pt", map_location=clf.device))
    proba = clf.predict_proba(entrada(olist))
    out = olist[["review_id", "order_id", "nota", "atrasado", "atraso_dias", "regiao", "categoria"]].copy()
    for j, a in enumerate(ASPECTS):
        for k, pol in enumerate(POLARITIES):
            out[f"p_{a}_{pol}"] = proba[:, j, k]
    out.to_csv(arq_olist, index=False)
    del clf; torch.cuda.empty_cache()
olist_p = pd.read_csv(arq_olist)
y = olist_p.atrasado.values
print(f"n = {len(olist_p)}, atrasados = {y.sum()}")
print("AUC P(entrega neg):", round(roc_auc_score(y, olist_p.p_entrega_negativo), 3),
      "| nota:", round(roc_auc_score(y, -olist_p.nota), 3),
      "| placebo P(produto neg):", round(roc_auc_score(y, olist_p.p_produto_negativo), 3))

In [ ]:
# 5) H2 — o Qwen2.5-7B rotula a validação (599) e o treino (5.589). Parte longa; retomável pelo cache no Drive
from absa.llm_labeler import hf_generator, label_texts, select_fixed_examples

MODELO = "Qwen/Qwen2.5-7B-Instruct"
exemplos = select_fixed_examples(demo, k=6)  # exemplos do prompt saem só do demo
ja_rotulado = all((SAIDA / f"repro_{n}_qwen7b.csv").exists() for n in ["valdev", "treino"])
generate = None if ja_rotulado else hf_generator(MODELO, load_in_4bit=True, max_new_tokens=60)  # baixa ~15 GB

def rotular(df, nome, lote=16):
    arq = SAIDA / f"repro_{nome}_qwen7b.csv"
    if arq.exists():
        print(f"✓ {nome} já rotulado — lendo do Drive"); return pd.read_csv(arq)
    t0 = time.time()
    rot, _ = label_texts(df.texto_modelo.tolist(), generate, MODELO, exemplos,
                         cache_name=f"qwen7b_{nome}", batch_size=lote)
    out = df[["review_id"]].copy()
    for a in ASPECTS:
        out[f"asp_{a}"] = [r[a] if r else None for r in rot]
    out["valido"] = [r is not None for r in rot]
    out.to_csv(arq, index=False)
    print(f"{nome}: {out.valido.sum()}/{len(out)} respostas válidas ({(time.time() - t0) / 60:.1f} min)")
    return out

def sem_invalidos(rot):
    pred = rot[LABEL_COLUMNS].copy()
    pred.loc[~rot.valido.to_numpy(), LABEL_COLUMNS] = "ausente"  # inválido conta como erro
    return pred.fillna("ausente")

t0 = time.time()
rot_val = rotular(val_dev, "valdev")
seg_por_texto = (time.time() - t0) / len(val_dev)
f1, lo, hi = bootstrap_ci(val_dev, sem_invalidos(rot_val))
print(f"Concordância LLM × humano na validação: F1 de pares {f1:.3f} (IC95% {lo:.3f}–{hi:.3f})")
display(per_aspect_report(val_dev, sem_invalidos(rot_val)).round(3))

rot_treino = rotular(treino, "treino")
arq_qual = SAIDA / "qualidade_rotulos_qwen7b.json"
if not arq_qual.exists():
    json.dump({"f1_pares_valdev": f1, "ic_valdev": [lo, hi], "f1_pares_treino": tuple_f1(treino, sem_invalidos(rot_treino)),
               "validos_treino": int(rot_treino.valido.sum()), "seg_por_texto": seg_por_texto, "modelo": MODELO,
               "exemplos_prompt": len(exemplos)}, open(arq_qual, "w"), indent=1, default=float)
del generate; gc.collect(); torch.cuda.empty_cache()  # libera a GPU para o BERTimbau

In [ ]:
# 6) H2 — BERTimbau treinado com os rótulos do LLM (L, 3 sementes); época escolhida no val_dev humano, como em H
treino_llm = treino[["review_id", "texto_modelo"]].merge(rot_treino[rot_treino.valido], on="review_id")
print(f"treino com rótulos do LLM: {len(treino_llm)} textos")
for seed in [13, 42, 87]:
    treinar("L", seed, entrada(treino_llm), treino_llm, entrada(val_dev), val_dev)

In [ ]:
# 7) Resumo no teste humano (média ± desvio entre sementes)
linhas = []
for arq in sorted(SAIDA.glob("pred_teste_bertimbau_*.csv")):
    p = pd.read_csv(arq).set_index("review_id").loc[teste.review_id, LABEL_COLUMNS].reset_index(drop=True)
    cond, seed = arq.stem.split("_")[-2], arq.stem.split("_")[-1]
    linhas.append({"cond": cond, "seed": seed, "f1_pares": tuple_f1(teste, p),
                   "f1_macro": macro_f1(teste, p), "f1_deteccao": detection_f1(teste, p)})
res = pd.DataFrame(linhas).groupby("cond")[["f1_pares", "f1_macro", "f1_deteccao"]].agg(["mean", "std"])
display(res.round(3))
m = res[("f1_pares", "mean")]
print(f"H2 — recuperação do gap (L − V)/(H − V) = {(m['L'] - m['V']) / (m['H'] - m['V']):.1%}")

In [ ]:
# 8) Baixar os resultados (resultados.zip)
import shutil
shutil.make_archive("/content/resultados", "zip", SAIDA)
if "google.colab" in sys.modules:
    from google.colab import files
    files.download("/content/resultados.zip")